In [13]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [14]:
X_train = np.load('X_train_w24.npy')
y_train = np.load('y_train_w24.npy')
X_val   = np.load('X_val_w24.npy')
y_val   = np.load('y_val_w24.npy')

print("X_train shape:", X_train.shape)
n_features = X_train.shape[2]
print("Number of input features:", n_features)

X_train shape: (82361, 24, 3)
Number of input features: 3


In [15]:
def build_lstm_model(window_size, n_features):
    model = keras.Sequential([
        layers.Input(shape=(window_size, n_features)),
        layers.LSTM(64, return_sequences=True),   # first LSTM layer
        layers.Dropout(0.2),
        layers.LSTM(32),                           # second LSTM layer
        layers.Dropout(0.2),
        layers.Dense(1)                             # predicted AC_POWER at t+1
    ])
    return model

model = build_lstm_model(window_size=24, n_features=n_features)

In [16]:
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 24, 64)         │        17,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,857 (116.63 KB)

 Trainable params: 29,857 (116.63 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint('best_lstm_model.keras', monitor='val_loss', save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

In [18]:
model.save('lstm_model_w24_untrained.keras')
files.download('lstm_model_w24_untrained.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
for window_size in [32, 48]:
    X_train_w = np.load(f'X_train_w{window_size}.npy')
    n_features = X_train_w.shape[2]

    model_w = build_lstm_model(window_size=window_size, n_features=n_features)
    model_w.compile(optimizer='adam', loss='mse', metrics=['mae'])

    print(f"\n--- Model for window size {window_size} ---")
    model_w.summary()

    model_w.save(f'lstm_model_w{window_size}_untrained.keras')
    files.download(f'lstm_model_w{window_size}_untrained.keras')


--- Model for window size 32 ---


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                   │ (None, 32, 64)         │        17,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 32, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,857 (116.63 KB)

 Trainable params: 29,857 (116.63 KB)

...

 Total params: 29,857 (116.63 KB)

 Trainable params: 29,857 (116.63 KB)

 Non-trainable params: 0 (0.00 B)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>